# 🔬 Shrink or Sink — Architecture Search Notebook
> **Runtime → Change runtime type → T4 GPU** before running!

This notebook runs the full search pipeline:
1. Install dependencies
2. Clone the repo
3. Show the ranked architecture search space
4. Train the Ultimate Teacher (ResNet-50 + Pseudo-Labeling)
5. Run binary search to find smallest model ≥ 85%
6. Inspect and save results

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'Device         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 📦 Step 1 — Clone Repo & Install Deps

In [7]:
!rm -rf shrink_or_sink_25
!git clone https://github.com/ayushaanand/shrink_or_sink_25.git
%cd shrink_or_sink_25
!pip install torch torchvision tqdm -q


fatal: destination path 'sos' already exists and is not an empty directory.


## 📊 Step 2 — Define Search Bounds


In [ ]:
# The search operates between lower and upper channel bounds.
# It probes the midpoint between lo and hi until convergence.
from dynamic_model import DynamicNet, size_mb, param_count

lo = [8, 16, 32, 64]
hi = [64, 128, 256, 256]

print(f"Search Space Bounds:\n")
print(f"  lo: {lo} -> {size_mb(DynamicNet(lo)):.4f} MB ({param_count(DynamicNet(lo)):,} params)")
print(f"  hi: {hi} -> {size_mb(DynamicNet(hi)):.4f} MB ({param_count(DynamicNet(hi)):,} params)")


## 🎓 Step 3 — Train the Ultimate Teacher (ResNet-50)
The teacher is **unconstrained in size** — it is never submitted.
Its only job is to provide soft-label knowledge to our tiny student.

This will run for 200 epochs across 3 phases (Burn-in, Pseudo-Labeling, Mastery).
> *Tip: It saves to Drive every single epoch. If Colab crashes, just run this cell again to resume!*


In [2]:
# Mount Drive to persist the teacher checkpoint
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/sos_checkpoints'
!mkdir -p {DRIVE_DIR}

Mounted at /content/drive


In [3]:
# ── Colab Resumable Teacher Training ────────────────────────────────
import os
TEACHER_PATH = f'{DRIVE_DIR}/teacher.pth'

# The script will automatically look for teacher_latest.pth in {DRIVE_DIR} to resume!
if os.path.exists(TEACHER_PATH):
    print('✅ Teacher already finalized and saved. Skipping.')
else:
    !python teacher_train.py \
        --data /content/data \
        --out {TEACHER_PATH} \
        --checkpoint {DRIVE_DIR}/teacher_latest.pth


✅ Teacher already trained. Skipping.


## 🛑 STOP HERE FOR NOW

> Since we are focusing exclusively on training the **Ultimate Teacher** (200 Epochs), you do not need to run the Search phase yet.
> 
> You can just leave this Colab tab open, let the cell above run for the next few hours, and check back periodically. It will autosave `teacher_latest.pth` to your Google Drive every epoch.

In [5]:
STUDENT_OUT = f'{DRIVE_DIR}/best_student.pth'

import os
if not os.path.exists(TEACHER_PATH):
    print('🛑 STOP: Teacher is not finished training yet! Wait for Phase 3 to complete.')
else:
    !python search.py \
        --data /content/data \
        --teacher {TEACHER_PATH} \
        --lo 8 16 32 64 \
        --hi 64 128 256 256 \
        --proxy-epochs 20 \
        --full-epochs 100 \
        --proxy-thresh 0.65 \
        --target-acc 0.85 \
        --out {STUDENT_OUT}


python3: can't open file '/content/search.py': [Errno 2] No such file or directory


## 📋 Step 5 — Inspect Search Results

In [ ]:
import json

with open('search_results.json') as f:
    log = json.load(f)

print(f"{'Iter':<6} {'Config':<30} {'MB':>8} {'Proxy':>8} {'Full':>8}  {'Verdict'}")
print('─' * 85)
for r in log:
    full = f"{r['full_acc']:.4f}" if r['full_acc'] is not None else '   —  '
    print(f"{r['iteration']:<6} {str(r['config']):<30} "
          f"{r['mb']:>8.4f} {r['proxy_acc']:>8.4f} {full:>8}   {r['verdict']}")


## ✅ Step 6 — Final Size & Accuracy Check

In [ ]:
import os, torch
from torchvision import transforms
from torchvision.datasets import STL10
from torch.utils.data import DataLoader
from dynamic_model import DynamicNet, ranked_configs, size_mb
from train_recipe import VAL_TRANSFORM, evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Find the best config from log
best = [r for r in log if r['verdict'] == 'sufficient']
best.sort(key=lambda r: r['mb'])

if not best:
    print('❌ No sufficient config found. Expand the grid or increase full-epochs.')
else:
    winner = best[0]
    print(f"Winner : {winner['config']}")
    print(f"Size   : {winner['mb']:.4f} MB")
    print(f"Val Acc: {winner['full_acc']:.4f} ({winner['full_acc']*100:.2f}%)")
    print(f"{'✅ Above 85%!' if winner['full_acc'] >= 0.85 else '❌ Below 85%'}")

    # Verify on fresh loader
    val_ds = STL10(root='./data', split='test', download=True, transform=VAL_TRANSFORM)
    val_ld = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2)
    student = DynamicNet(winner['config']).to(device)
    student.load_state_dict(torch.load(STUDENT_OUT, map_location=device))
    acc = evaluate(student, val_ld, device)
    print(f'\n📊 Live verified accuracy: {acc:.4f} ({acc*100:.2f}%)')

    sz = os.path.getsize(STUDENT_OUT) / 1024**2
    print(f'📦 Actual .pth file size : {sz:.4f} MB')